# Step 07 — POI classification

Step 05 asked the model what happens in each **building**; step 06 put those
answers on the polygons. But a building's labels say *which* activities happen
somewhere inside, not *which occupant* serves which. A building labelled
`retail_daily; leisure` with a bakery at 30 % and a pub at 70 % cannot be
split without knowing that the bakery is the retail and the pub is the
leisure. This notebook asks the model, for every POI placed in a building,
**the purposes people come to that occupant for** — the same twelve MiD
labels, the same definitions — so that the redistribution (step 08) can
weight `volume × share_in_building × label`.

| | |
|---|---|
| **Reads** | `data/output/06_buildings_classified.gpkg` — layers `buildings` and `building_pois`; `data/output/01_all_pois.gpkg` for the OSM key of each POI's use and the folded `tags` (brand, operator); `.env` for the API token (git-ignored, never printed) |
| **Writes** | `data/output/07_llm_poi_input.parquet` — one record per POI, the text the model reads; `07_llm_poi_sample_answers.jsonl` — the sample run; the full run's `07_llm_poi_answers.jsonl` is written by `scripts/07_run_llm_pois.py` on the Linux server |
| **Needs** | `pyogrio`, `pandas`, `pyarrow`, `requests` |

## State of this notebook

Built **one step at a time**, each run and inspected before the next is written.

| step | | status |
|---|---|---|
| **07.1** | **Input contract** — the two layers of step 06 and the POI layer of step 01, checked against each other | **implemented** |
| **07.2** | **The records** — one per own pair: the occupant, where it sits, the building around it | **implemented** |
| **07.3** | **The prompt** — the building prompt rewritten for one occupant; the label definitions asserted identical | **implemented** |
| **07.4** | **The call list** — every own pair, no dedupe: 22,633 calls | **implemented** |
| **07.5** | **Before the calls** — the prompt review gate, the stand-in dry run, then the sample on the real model | **implemented — sample run; the full run is `scripts/07_run_llm_pois.py`** |
| **07.6** | **Assembly** — the answers joined onto `building_pois`, work by rule, the disagreement table against the building labels | pending (after the Linux run, on Windows) |

## Decisions (2026-09-30, `docs/07-poi-classification-handoff.md`)

* **The unit of one call is one POI in one building** — an own pair of
  `building_pois` (`how` = inside, snap, unit). The 18,652 `site` rows are not
  asked: a site's label is its buildings', already in `mid_labels`.
* **Only the POIs that share their building are asked** (2026-10-01): the
  8,091 own pairs with `share_in_building < 1`, on the 2,915 buildings with two
  or more own POIs. The 14,542 sole occupants (share 1) and the buildings
  without own POIs already have their labels from step 05. Of the 8,091,
  227 are left out by `config.LLM_POI_NOT_ASKED` (also 2026-10-01): 183
  `landuse=*` areas — zoning polygons that step 04.5 attached to a building
  like a POI; they describe the ground, not an occupant — and 44 vacant
  units with no activity to label. Step 08 uses `share_in_building` as
  step 04.5 computed it: an excluded pair's share carries no activity. No dedupe among them: every POI is answered in its own
  building's context and there is no signature step in the assembly. The
  input file is the plan: every `poi_id` in it is asked.
* **Same labels, same definitions, no Bosserhof class.** The Bosserhof class
  is a building property and already drives the Workers weight. `work` is
  added by rule afterwards, as in step 06: every occupant is a workplace.
* **Same model, endpoint, client and run** as step 05, keyed by `poi_id`
  (`lib/llm_client`, `lib/llm_run` take the prompt, the schema and the id
  column as arguments now; step 05 keeps its defaults).
* **The prompt is reviewed and confirmed before any call** (section 5).


In [ ]:
import os, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError('Cannot find the pipeline root (the folder containing config.py). '
                       f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.')
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import json
import re
import time
import numpy as np
import pandas as pd
import pyogrio

from config import (
    CLASSIFIED_BUILDINGS_FILE, ALL_POIS_FILE, OUTPUT_DIR,
    LLM_SYSTEM_PROMPT, LLM_POI_SYSTEM_PROMPT, LLM_ACTIVITY_LABELS, WORK_IMPLIED_BY, LLM_CONFIDENCE_LEVELS,
    LLM_OUTPUT_SCHEMA, LLM_POI_OUTPUT_SCHEMA, LLM_POI_NAME_FALLBACK_TAGS,
    LLM_POI_QUALIFIER_KEYS, LLM_POI_BUILDING_QUALIFIER_FOR, LLM_POI_ASK_SHARE_BELOW, LLM_POI_NOT_ASKED,
    LLM_POI_INPUT_FILE, LLM_POI_RECORD_SAMPLE_PER_KIND,
    LLM_API_URL, LLM_MODEL, LLM_MAX_ATTEMPTS,
    LLM_POI_SAMPLE_IDS, LLM_POI_SAMPLE_ANSWERS_FILE, LLM_POI_ANSWERS_FILE, LLM_POI_STATUS_FILE,
)
from lib.checks import require_file, require_non_empty, require_unique, require_cols
from lib.llm_record import build_poi_records, poi_qualifiers, POI_LINE_PREFIXES, GROUP_SAME_USE_FROM
from lib.llm_client import read_token, prompt_sha, parse_answer, validate_answer, InvalidAnswer
from lib.llm_run import run, load_answers, valid_answers, answer_columns

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 70)

print('Root :', ROOT_DIR)
print('Input:', CLASSIFIED_BUILDINGS_FILE.name, '+', ALL_POIS_FILE.name)


## 1. Input contract

Three things must agree with each other:

* the **buildings** of step 06 — every own pair must point at one of them; the
  record reads the cadastre class and name, the municipality, the footprint
  and the height from this layer (the slim column set of step 06; the OSM
  footprint tag and the land use of step 04 are not carried, and the POI
  record does without them);
* the **building–POI pairs** of step 04.5, carried unchanged into step 06 —
  the own pairs (`how` ≠ `site`) are the calls; `poi_id` must be unique among
  them, and `share_in_building` must sum to 1.0 on every building that has
  own pairs, or the split in step 08 leaks demand;
* the **POI layer** of step 01 — for the OSM key of each use (`poi_use_tag`;
  without it a tag is written as the bare value, which the step-05 dry run
  showed the model cannot read) and the folded `tags`, from which an unnamed
  occupant's `brand` or `operator` is taken. Every pair's `poi_id` must exist
  there with the same `poi_use`, or the two files are from different runs.

The building's own answer (`mid_labels`, `llm_type`) is read here for the
review printouts only. It **never enters a record**: shown, it would anchor
the model to the building's labels, and the point of this step is to find out
which occupant serves which of them.


In [ ]:
require_file(CLASSIFIED_BUILDINGS_FILE, 'classified buildings (step 06)')
require_file(ALL_POIS_FILE, 'POIs (step 01)')
_layers = [l[0] for l in pyogrio.list_layers(CLASSIFIED_BUILDINGS_FILE)]
for _l in ('buildings', 'building_pois'):
    if _l not in _layers:
        raise AssertionError(f'layer "{_l}" missing from {CLASSIFIED_BUILDINGS_FILE.name}: {_layers}')

# --- the buildings: what the record reads, plus the building's answer for the review ---
_BLD_RECORD_COLS = ['building_id', 'source', 'label_en', 'name', 'city', 'area_m2', 'height_top_max_m']
_BLD_REVIEW_COLS = ['n_pois', 'n_sites', 'mid_labels', 'llm_type', 'llm_confidence', 'volume_3d_m3']
bld = pyogrio.read_dataframe(CLASSIFIED_BUILDINGS_FILE, layer='buildings', read_geometry=False,
                             columns=_BLD_RECORD_COLS + _BLD_REVIEW_COLS)
require_non_empty(bld, 'buildings')
require_unique(bld, 'building_id', 'buildings')
if bld['mid_labels'].isna().any() or (bld['mid_labels'] == '').any():
    raise AssertionError('a building without labels - step 06 is incomplete')

# --- the pairs ------------------------------------------------------------------------
pairs = pyogrio.read_dataframe(CLASSIFIED_BUILDINGS_FILE, layer='building_pois', read_geometry=False)
require_cols(pairs, ['building_id', 'poi_id', 'poi_role', 'how', 'poi_use', 'name', 'poi_parent_id',
                     'parent_use', 'parent_name', 'share_in_building', 'share_of_site', 'snap_m'], 'building_pois')
_orphans = set(pairs['building_id']) - set(bld['building_id'])
if _orphans:
    raise AssertionError(f'{len(_orphans):,} building_pois rows point at buildings not in the layer, e.g. {sorted(_orphans)[:3]}')
own = pairs[pairs['how'] != 'site']
require_unique(own, 'poi_id', 'own pairs')
if (own['how'] == 'site').any() or (own['poi_role'] == 'site').any():
    raise AssertionError('a site row among the own pairs')
_sums = own.groupby('building_id')['share_in_building'].sum()
if own['share_in_building'].isna().any() or (_sums.sub(1.0).abs() > 1e-6).any():
    raise AssertionError(f"share_in_building does not sum to 1.0 on {int((_sums.sub(1.0).abs() > 1e-6).sum()):,} buildings with own pairs")
print(f'  ok  {len(pairs):,} pairs on {pairs["building_id"].nunique():,} buildings: '
      f'{len(own):,} own pairs ({own["how"].value_counts().to_dict()}) on {own["building_id"].nunique():,} buildings, '
      f'{int((pairs["how"] == "site").sum()):,} site memberships; shares sum to 1.0 on every building with own pairs')

# --- the POI layer: the OSM key of the use, the name fallbacks, the qualifiers, the geometry kind ---
_pois = pyogrio.read_dataframe(ALL_POIS_FILE, layer='pois', read_geometry=False)
require_unique(_pois, 'poi_id', 'POIs')
require_cols(_pois, ['poi_use', 'poi_use_tag', 'geom_kind', 'tags'], 'POIs')
def _tag_value(tags, key):
    try:
        return json.loads(tags).get(key) if isinstance(tags, str) and tags.strip() else None
    except json.JSONDecodeError:
        return None
for _k in LLM_POI_NAME_FALLBACK_TAGS:
    if _k not in _pois.columns:
        _pois[_k] = _pois['tags'].map(lambda t, _k=_k: _tag_value(t, _k))
_pois['qualifiers'] = poi_qualifiers(_pois, LLM_POI_QUALIFIER_KEYS, LLM_POI_BUILDING_QUALIFIER_FOR)
_keys = _pois[['poi_id', 'poi_use', 'poi_use_tag', 'geom_kind', 'qualifiers'] + list(LLM_POI_NAME_FALLBACK_TAGS)]
pairs = pairs.merge(_keys.rename(columns={'poi_use': '_poi_use_01'}), on='poi_id', how='left', validate='many_to_one')
if pairs['poi_use_tag'].isna().any():
    raise AssertionError(f'{int(pairs["poi_use_tag"].isna().sum()):,} building_pois rows have no POI in {ALL_POIS_FILE.name}')
if (pairs['poi_use'] != pairs['_poi_use_01']).any():
    raise AssertionError('poi_use differs between step 01 and step 06 for the same poi_id - the two files are from different runs')
pairs = pairs.drop(columns='_poi_use_01')
# the parent's key, for the role line of the unit rows
pairs = pairs.merge(_keys[['poi_id', 'poi_use_tag']].rename(columns={'poi_id': 'poi_parent_id', 'poi_use_tag': 'parent_use_tag'}),
                    on='poi_parent_id', how='left', validate='many_to_one')
own = pairs[pairs['how'] != 'site']
_unnamed = own['name'].isna()
print(f'  ok  every pair has its OSM key ({pairs["poi_use_tag"].nunique()} keys); '
      f'{int(_unnamed.sum()):,} own pairs are unnamed, {int((_unnamed & own["brand"].notna()).sum()):,} of them carry a brand, '
      f'{int((_unnamed & own["brand"].isna() & own["operator"].notna()).sum()):,} an operator; '
      f'{int((own["qualifiers"] != "").sum()):,} carry a qualifying tag')
_u = own[own['how'] == 'unit']
_outline = (own['poi_use_tag'] == 'building') | (own['poi_id'] == own['building_id'])
print(f'  ..  units: {len(_u):,}, parent tag known for {int(_u["parent_use_tag"].notna().sum()):,}, '
      f'parent named for {int(_u["parent_name"].notna().sum()):,}, parent is the building\'s outline for {int((_u["parent_use_tag"] == "building").sum()):,}; '
      f'snapped: {int((own["how"] == "snap").sum()):,} ({own.loc[own["how"] == "snap", "geom_kind"].value_counts().to_dict()}), '
      f'median {own.loc[own["how"] == "snap", "snap_m"].median():.0f} m; '
      f'the building\'s own OSM outline: {int(_outline.sum()):,} own pairs')

# --- who is asked: the POIs that share their building (config.LLM_POI_ASK_SHARE_BELOW), minus LLM_POI_NOT_ASKED
_sole = own['share_in_building'] >= LLM_POI_ASK_SHARE_BELOW - 1e-9
_not_asked = pd.Series(False, index=own.index)
for _k, _vals in LLM_POI_NOT_ASKED.items():
    _not_asked |= (own['poi_use_tag'] == _k) & (own['poi_use'].isin(_vals) if _vals else True)
_asked = ~_sole & ~_not_asked
_nsib = own.groupby('building_id')['poi_id'].transform('size')
if ((~_sole) != (_nsib > 1)).any():
    raise AssertionError('share_in_building < 1 and "has other own POIs on the building" disagree - the shares are not what step 04.5 promised')
print(f'  ok  {int(_asked.sum()):,} own pairs are asked: the POIs on the {own.loc[_asked, "building_id"].nunique():,} buildings with two or more own POIs; '
      f'{int(_sole.sum()):,} sole occupants (share 1) keep their building\'s labels'
      + (f'; {int((_not_asked & ~_sole).sum()):,} left out by config.LLM_POI_NOT_ASKED ({own.loc[_not_asked & ~_sole, "poi_use_tag"].value_counts().to_dict()})'
         if (_not_asked & ~_sole).any() else '; nothing left out by config.LLM_POI_NOT_ASKED'))


## 2. The records

One POI, one record, one call. The record is the building record of step 05
with the occupant pulled out in front, so the model cannot mistake context for
the thing to label:

```
occupant: "Fielmann" (shop=optician)
role: inside the building
building also inside: "Delphin-Palast" (amenity=cinema); "Die Zahnarztpraxis Mitte" (amenity=dentist); "Eiscafe Roma" (amenity=cafe); "Kampe" (shop=jewelry); "Reinigung City Clean" (shop=dry_cleaning)
building site: unnamed (landuse=commercial)
building cadastre: Building for trade and services with housing, named "Kino"
building place: Wolfsburg | footprint 1816 m2 | height 13 m
```

```
occupant: unnamed (sport=gymnastics)
role: inside the building
building footprint: building=sports_centre, named "Turnhalle"
building cadastre: Sports hall, gymnasium
building place: Helmstedt | footprint 530 m2 | height 14 m
```

```
occupant: unnamed, brand "Allianz" (office=insurance)
role: placed on the nearest building: a point mapped 1 m outside it
building cadastre: residential buildings
building place: Wolfsburg | footprint 75 m2 | height 10 m
```

* **`occupant`** — the name in quotes and the OSM tag as `key=value`. An
  unnamed occupant shows its `brand`, else its `operator`, marked as such
  (196 of the 2,474 unnamed own POIs carry one): a name is where the model's
  world knowledge works. Otherwise `unnamed`. A use that step 01 wrote as the
  key itself (the mapper gave only `yes`: 242 own pairs, `historic`, `office`,
  `shop`, `club` …) is written `key=yes`, not `historic=historic`. Qualifying
  tags follow after a comma (`LLM_POI_QUALIFIER_KEYS`): `sport=multi,
  leisure=sports_centre, building=sports_hall` says hall rather than pitch,
  `amenity=social_facility, social_facility=group_home,
  social_facility:for=child;juvenile` says which kind of facility — the use
  alone hid this on 433 of the 548 sport POIs and 326 of the 374 social
  facilities.
* **`role`** — where it sits: `inside the building`; `the building itself:
  the tag is the one on its OSM outline` for a POI that *is* the building's
  outline (`building=*`, 1,534 own pairs with a median area ratio of 1.01 to
  the footprint, or the 519 OSM gap buildings whose polygon is the POI); `a
  unit inside "<parent>" (<tag>)` for the units of step 04.5, the parent's key
  joined from step 01 — but simply `inside the building` when the parent is
  the building's own outline or the occupant's own outline; and for the
  snapped POIs `placed on the nearest building: a point mapped N m outside
  it` or `… an area mapped N m from it` (903 of the 2,004 are areas, mostly
  larger than the building).
* **The building lines** are `build_record`'s lines minus the occupant, in the
  same order, each prefixed `building`: `also inside` (the other own POIs,
  written exactly as the occupant line — quoted, brand or operator for an
  unnamed one — ordered by share, grouped from `GROUP_SAME_USE_FROM` of one
  tag and listed as often as they occur, the parent of a unit left out
  because the role line has it), `footprint` (the building's own OSM
  outline(s) with tag and name, instead of listing them as tenants), `site`,
  `cadastre`, `place`. No `land` line: step 06 does not carry it. The
  building's own labels are never in the record.
* The same cleaning rules as step 05: whole metres and square metres without
  thousands separators, no line breaks or semicolons in a name, the
  municipality without its legal-form suffix. Every line starts with one of
  `POI_LINE_PREFIXES` — a check below enforces it.

The rendering lives in `lib/llm_record.py` (`build_poi_record`,
`build_poi_records`, `poi_qualifiers`), free of call logic, so the validation
can re-render what the model saw. The output file carries `poi_id,
building_id, how, evidence, record, n_chars`; `evidence` says what the
occupant line carries (`named`, `brand`, `operator`, `unnamed`). With no
dedupe this file is also the plan; `config.LLM_POI_NOT_ASKED` (empty by
decision) is the one way to leave a kind of own pair out of it.


In [ ]:
print('Rendering one record per asked pair ...', flush=True)
t0 = time.perf_counter()
records = build_poi_records(pairs, bld[_BLD_RECORD_COLS], poi_ids=own.loc[_asked, 'poi_id'])
print(f'  ok  {len(records):,} records  [{time.perf_counter() - t0:,.0f}s]')
require_unique(records, 'poi_id', 'records')
if len(records) != int(_asked.sum()) or set(records['poi_id']) != set(own.loc[_asked, 'poi_id']):
    raise AssertionError('the records do not cover exactly the asked pairs')
if not records['record'].str.contains('\nbuilding also inside: |\nbuilding footprint: |\nrole: a unit inside ').all():
    raise AssertionError('an asked POI has no other occupant in its record (neither beside it nor as its parent) - it should be a sole occupant')
_stray = [l for rec in records['record'] for l in rec.split('\n') if not l.startswith(POI_LINE_PREFIXES)]
if _stray:
    raise AssertionError(f'{len(_stray)} record lines start with no known prefix, e.g. {_stray[:3]!r}')
if not records['record'].str.startswith('occupant: ').all() or (records['n_chars'] < 60).any():
    raise AssertionError('a record without an occupant line, or nearly empty')
print(f'  ok  every line of every record starts with one of {POI_LINE_PREFIXES}')
print()
print('  ..  record length (characters):')
print(records['n_chars'].describe(percentiles=[.5, .9, .99]).round(0).astype(int).to_string())
_longest = records.sort_values('n_chars', ascending=False).head(3)
print('  ..  the three longest records belong to: '
      + ', '.join(f'{r.poi_id} in {r.building_id} ({r.n_chars:,} chars)' for r in _longest.itertuples()))
print()
print('  ..  what the occupant line carries, by how the POI sits on the building:')
print(pd.crosstab(records['evidence'], records['how'], margins=True).to_string())
_lines = records['record'].str.split('\n').map(lambda ls: [l.split(':', 1)[0] for l in ls])
print('  ..  lines present:', pd.Series([p for ls in _lines for p in ls]).value_counts().to_dict())

# --- read some, one kind at a time --------------------------------------------------
_rng = np.random.default_rng(7)
for (_how, _ev), _grp in records.groupby(['how', 'evidence']):
    print()
    print(f'=== {_how} / {_ev} ({len(_grp):,} POIs) - {min(LLM_POI_RECORD_SAMPLE_PER_KIND, len(_grp))} at random:')
    for _i in _rng.choice(_grp.index, size=min(LLM_POI_RECORD_SAMPLE_PER_KIND, len(_grp)), replace=False):
        print(records.at[_i, 'record'])
        print('-' * 60)

# --- write: the asked pairs are the plan ----------------------------------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
records = records[['poi_id', 'building_id', 'how', 'evidence', 'record', 'n_chars']]
records.to_parquet(LLM_POI_INPUT_FILE, index=False)
_back = pd.read_parquet(LLM_POI_INPUT_FILE)
if len(_back) != len(records) or list(_back.columns) != list(records.columns) or not (_back['record'] == records['record']).all():
    raise AssertionError('the parquet file does not read back as written')
print()
print(f'  ok  {len(_back):,} rows x {len(_back.columns)} columns -> {LLM_POI_INPUT_FILE.name} '
      f'({LLM_POI_INPUT_FILE.stat().st_size / 1e6:.1f} MB); columns: {list(_back.columns)}')


## 3. The prompt

`LLM_POI_SYSTEM_PROMPT` is `LLM_SYSTEM_PROMPT` rewritten for one occupant, by
the same rules as the building prompt — definitions, not examples;
understanding, not clues; no scope text. Presented for review as a marked
change against the building prompt, in three groups (hand-off §4.1):

1. **Removed** — Part B (the whole Bosserhof section), the "You must produce
   TWO outputs" framing, `bosserhof_class` in the output JSON and in the
   closing instructions, the "in Part B it separates subcategories" clause.
2. **Kept verbatim** — the ALLOWED ACTIVITY LABELS list and the LABEL
   DEFINITIONS block. The cell below asserts byte equality with the building
   prompt, so POI labels mean exactly what building labels mean, and neither
   prompt can drift alone.
3. **Changed or new** — the opening role sentence (one occupant instead of
   one building); the description of the record lines (`occupant`, `role`,
   the `building …` lines, replacing the six building-record sources); the
   rule that the building lines locate the occupant and never add a purpose,
   except the institution rule; the PROCEDURE for one occupant; the output
   JSON without `bosserhof_class` and a shorter reason; the confidence tiers
   for an occupant.

The sha of the confirmed text is what every answer carries; changing it later
invalidates the run.


In [ ]:
print(LLM_POI_SYSTEM_PROMPT)
print('=' * 78)

_DIV = '─' * 36
def _block(prompt, start, end):
    head = f'{_DIV}\n{start}\n{_DIV}\n'
    a = prompt.index(head) + len(head)
    return prompt[a:prompt.index(f'{_DIV}\n{end}\n{_DIV}\n')]
_defs_building = _block(LLM_SYSTEM_PROMPT, 'LABEL DEFINITIONS', 'PART B — BOSSERHOF BUILDING-USE CLASS')
_defs_poi = _block(LLM_POI_SYSTEM_PROMPT, 'LABEL DEFINITIONS', 'OUTPUT FORMAT (STRICT JSON ONLY)')
if _defs_building != _defs_poi:
    raise AssertionError('the LABEL DEFINITIONS block of the POI prompt differs from the building prompt')

def _labels_in(prompt):
    lab_block = prompt.split('ALLOWED ACTIVITY LABELS', 1)[1].split('PROCEDURE:', 1)[0]
    return tuple(l[2:].strip() for l in lab_block.splitlines() if l.startswith('- '))
if not _labels_in(LLM_POI_SYSTEM_PROMPT) == _labels_in(LLM_SYSTEM_PROMPT) == LLM_ACTIVITY_LABELS:
    raise AssertionError(f'activity labels differ - POI prompt {_labels_in(LLM_POI_SYSTEM_PROMPT)} vs config {LLM_ACTIVITY_LABELS}')
for _lab in LLM_ACTIVITY_LABELS:
    if f'\n- {_lab}\n' not in _defs_poi:
        raise AssertionError(f'label "{_lab}" has no definition block in the POI prompt')
_conf = tuple(re.findall(r'^- (\w+)\s+→', LLM_POI_SYSTEM_PROMPT.rsplit('confidence:', 1)[1], re.M))
if _conf != LLM_CONFIDENCE_LEVELS:
    raise AssertionError(f'confidence levels differ - prompt {_conf} vs config {LLM_CONFIDENCE_LEVELS}')
if 'osserhof' in LLM_POI_SYSTEM_PROMPT or 'PART B' in LLM_POI_SYSTEM_PROMPT:
    raise AssertionError('the POI prompt still mentions the Bosserhof class')
if set(LLM_POI_OUTPUT_SCHEMA['properties']) != set(LLM_POI_OUTPUT_SCHEMA['required']) != set():
    raise AssertionError('every field of the POI answer is required')
if set(LLM_POI_OUTPUT_SCHEMA['properties']) != set(LLM_OUTPUT_SCHEMA['properties']) - {'bosserhof_class'}:
    raise AssertionError('the POI schema must be the building schema minus bosserhof_class')
if LLM_POI_OUTPUT_SCHEMA['properties']['mid_labels'] != LLM_OUTPUT_SCHEMA['properties']['mid_labels']:
    raise AssertionError('mid_labels differ between the two schemas')
if WORK_IMPLIED_BY != set(LLM_ACTIVITY_LABELS) - {'work'}:
    raise AssertionError('WORK_IMPLIED_BY must be every label but work')

print(f'  ok  the LABEL DEFINITIONS block ({len(_defs_poi):,} characters) and the {len(LLM_ACTIVITY_LABELS)} labels are the building prompt\'s, byte for byte; '
      f'{len(LLM_CONFIDENCE_LEVELS)} confidence levels; no Bosserhof class')
print(f'  ok  work by rule for any of: {", ".join(sorted(WORK_IMPLIED_BY))}')
print(f'  ..  POI prompt {prompt_sha(LLM_POI_SYSTEM_PROMPT)}: {len(LLM_POI_SYSTEM_PROMPT):,} characters, {len(LLM_POI_SYSTEM_PROMPT.split()):,} words '
      f'(the building prompt {prompt_sha()}: {len(LLM_SYSTEM_PROMPT):,} characters); a record adds a median {int(records["n_chars"].median())} characters')
print()
print('  ..  the answer schema (every reply is validated against it):')
print(json.dumps(LLM_POI_OUTPUT_SCHEMA, indent=2, ensure_ascii=False))


## 4. The call list

Only the POIs that share their building are asked (decided 2026-10-01): the
own pairs with `share_in_building < 1`, which are exactly the POIs on
buildings with two or more own POIs. A POI with share 1 is the building's
only occupant — its record went into the building's call in step 05 and the
building's labels are its labels; a building without own POIs keeps the
building's labels as well. Both stay at building level, without a
call: the POI output of section 6 leaves them out. No dedupe among the asked: every pair is its own call,
answered in its own building's context, so there are no copied answers. The
input file **is** the plan — the run asks every `poi_id` in it and skips
whatever already has a valid answer under the current prompt. The size of
the run, on record:


In [ ]:
_calls = len(records)
print(f'  ..  {_calls:,} calls = the POIs on the {records["building_id"].nunique():,} buildings with two or more own POIs '
      f'({int(_sole.sum()):,} sole occupants and {int((~bld["building_id"].isin(own["building_id"])).sum()):,} buildings without own POIs keep the building\'s labels)')
print(f'  ..  by how the POI sits: {records["how"].value_counts().to_dict()}')
print(f'  ..  by what the occupant line carries: {records["evidence"].value_counts().to_dict()}')
_multi = bld['mid_labels'].str.split(';').map(lambda ls: len([l for l in ls if l != 'work']) >= 2)
_on_multi = records['building_id'].isin(bld.loc[_multi, 'building_id'])
print(f'  ..  {int(_on_multi.sum()):,} of them sit on buildings that carry two or more non-work labels - the ones the split in step 08 '
      f'needs most; the rest make a workplace-only occupant reduce its neighbours\' volume')
for _s, _what in ((8.0, 'the step-05 median per call'), (19.5, 'step 05 wall-clock per call, pauses included')):
    print(f'  ..  at {_s} s per call ({_what}): {_calls * _s / 3600:,.0f} hours = {_calls * _s / 86400:.1f} days, one request at a time')


## 5. Before the calls: the review gate, the stand-in, the sample

Three things happen before the run, in this order, and each blocks the next:

1. **The prompt review gate** (hand-off §4.1). The user reads the prompt as
   the marked change of section 3, says what to change, and only the
   confirmed text goes into `config.LLM_POI_SYSTEM_PROMPT`. No call is made
   before that.
2. **The stand-in dry run.** The sample records below are read cold — the
   prompt and one record, nothing else — by independent stand-ins, several
   per record, to find what the renderer hides from the model and where the
   prompt lets two readings through, before a single real call is spent. The
   step-05 dry run found the bare tag values and the thousands separators
   this way. **Done 2026-09-30 on the first draft**: 26 records, three cold
   readers each, plus a prompt critic and a renderer critic. Labels were
   identical across the three readers on 22 of 26 (ignoring `work`); the
   four splits were genuine boundary cases (variety store: `retail_daily`
   too or not; restaurant: `meetup` too or not; martial arts: `lessons`;
   a library in a school: `school` by the institution rule). 53 of the 78
   "unclear" notes were the same question — whether to give `work` to every
   staffed occupant — so the prompt now says when `work` applies. The
   critics found what the first renderer hid, each claim measured on all
   22,633 records and fixed in `lib/llm_record.py` (see section 2):
   `historic=historic` for `yes` values, hidden qualifying tags, the
   building's own outline listed as a tenant or as a unit's parent, snapped
   areas described as misplaced points, neighbours written differently from
   the occupant, a unit's parent repeated among the neighbours; and in the
   prompt: a second unannounced exception in step 3, overlapping confidence
   tiers, "what the occupant is part of", an unscoped "names carry world
   knowledge", the institution rule without saying where the institution is
   read from. Two findings are decisions rather than fixes and are left to
   the reviewer: the 464 landuse polygons placed as occupants (a zoning
   area, median seven times the building's footprint, taking over half of
   the share on 127 buildings with real occupants) and the 96 vacant units
   (`config.LLM_POI_NOT_ASKED`).
3. **The sample on the real model.** `LLM_POI_SAMPLE_IDS` — 26 asked POIs
   chosen to cover the record kinds: named chains and one-offs, brand and
   operator as the name, `key=yes` tags, unnamed with a generic and with a
   specific tag, units inside a mall, a supermarket, a hotel, a church and a
   faculty, snapped points and areas near and far, the building's own outline
   on the footprint line, a multi-value tag, an OSM gap building without a
   cadastre line, two libraries in schools (the institution rule), a social
   facility with qualifiers, the longest record, a parent and one of its
   units — asked once each into their own answers file, printed next to the
   building's labels for review. **No full run before the sample is
   reviewed** (the step-05 rule).

The first cell makes no call: it checks the token is there and exercises the
validator on replies a model could give, under the POI schema.


In [ ]:
read_token()
print('  ok  API token found in .env / environment (not shown)')
print(f'  ..  endpoint {LLM_API_URL}')
print(f'  ..  model {LLM_MODEL} | the POI system prompt and one record per call, nothing else | up to {LLM_MAX_ATTEMPTS} attempts per POI | prompt {prompt_sha(LLM_POI_SYSTEM_PROMPT)}')
print(f'  ..  answers file columns: {answer_columns("poi_id", LLM_POI_OUTPUT_SCHEMA)}')

_good = ('{"interpreted_type": "a bakery", "mid_labels": ["retail_daily"], "confidence": "high", "reason": "the occupant line"}')
_a = validate_answer(parse_answer('```json\n' + _good + '\n```'), LLM_POI_OUTPUT_SCHEMA)
if _a != {'interpreted_type': 'a bakery', 'mid_labels': ['retail_daily'], 'confidence': 'high', 'reason': 'the occupant line'}:
    raise AssertionError(_a)
_b = validate_answer(parse_answer(_good.replace('"confidence"', '"bosserhof_class": "Retail", "confidence"')), LLM_POI_OUTPUT_SCHEMA)
if 'bosserhof_class' in _b:
    raise AssertionError('a Bosserhof class in the reply must be dropped, not kept')
if validate_answer(parse_answer(_good.replace('"high"', '"High"')), LLM_POI_OUTPUT_SCHEMA)['confidence'] != 'high':
    raise AssertionError('letter case is normalised, nothing else')
print('  ok  a valid reply passes, also inside a code fence or with a stray bosserhof_class; letter case and whitespace are normalised')
for _bad, _why in [
    (_good.replace('["retail_daily"]', '[]'), 'empty label list'),
    (_good.replace('"confidence": "high", ', ''), 'a field missing'),
    (_good.replace('retail_daily', 'shopping'), 'a label outside the twelve'),
    ('The occupant is probably a bakery.', 'no JSON at all'),
]:
    try:
        validate_answer(parse_answer(_bad), LLM_POI_OUTPUT_SCHEMA)
        raise AssertionError(f'accepted a reply it must reject: {_why}')
    except InvalidAnswer as _e:
        print(f'  ok  rejected ({_why}): {_e}')


In [ ]:
# the sample records as the model will see them, with the building's answer beside them for the reviewer
_rec = dict(zip(records['poi_id'], records['record']))
_absent = [p for p in LLM_POI_SAMPLE_IDS if p not in _rec]
if _absent or len(set(LLM_POI_SAMPLE_IDS)) != len(LLM_POI_SAMPLE_IDS):
    raise AssertionError(f'sample POIs without a record, or listed twice: {_absent}')
_bld_by_id = bld.set_index('building_id')
_own_by_id = own.set_index('poi_id')
print(f'=== the {len(LLM_POI_SAMPLE_IDS)} sample records, with the building\'s step-05 answer (never in the record):')
for _p in LLM_POI_SAMPLE_IDS:
    _b = _bld_by_id.loc[_own_by_id.at[_p, 'building_id']]
    print()
    print(f'--- {_p} in {_own_by_id.at[_p, "building_id"]}  |  share {_own_by_id.at[_p, "share_in_building"]:.3f}  |  '
          f'building: {_b["mid_labels"]} ({_b["llm_confidence"]}) "{str(_b["llm_type"])[:60]}"')
    print(_rec[_p])


In [ ]:
# SAMPLE RUN - only after the prompt of section 3 has been confirmed (hand-off 4.1)
# resumable: a second execution of this cell asks only what has no valid answer yet
_summary = run(LLM_POI_SAMPLE_IDS, _rec, LLM_POI_SAMPLE_ANSWERS_FILE, label='sample', status_every_s=30,
               status_file=LLM_POI_STATUS_FILE,                  # never the step-05 status file
               id_col='poi_id', system_prompt=LLM_POI_SYSTEM_PROMPT, schema=LLM_POI_OUTPUT_SCHEMA)

_cols = answer_columns('poi_id', LLM_POI_OUTPUT_SCHEMA)
sample = valid_answers(load_answers(LLM_POI_SAMPLE_ANSWERS_FILE, _cols), prompt_sha(LLM_POI_SYSTEM_PROMPT), 'poi_id')
sample = sample[sample['poi_id'].isin(LLM_POI_SAMPLE_IDS)]
if len(sample) < len(LLM_POI_SAMPLE_IDS):
    raise AssertionError(f'{len(LLM_POI_SAMPLE_IDS) - len(sample)} sample POI(s) have no valid answer - run this cell again to resume')

print()
print(f'=== the real model on the {len(sample)} sample POIs, once each:')
print(f"  {sample['elapsed_s'].median():.1f} s per call (median), {sample['elapsed_s'].mean():.1f} mean, "
      f"{sample['elapsed_s'].min():.1f} min, {sample['elapsed_s'].max():.1f} max; "
      f"{int((sample['attempts'] > 1).sum())} of {len(sample)} calls needed a retry")
print(f"  confidence {sample['confidence'].value_counts().to_dict()} | "
      f"work as the only label on {sum(list(l) == ['work'] for l in sample['mid_labels'])} POI(s)")
_ans = sample.set_index('poi_id')
for _p in LLM_POI_SAMPLE_IDS:
    _a = _ans.loc[_p]
    _b = _bld_by_id.loc[_own_by_id.at[_p, 'building_id']]
    _poi_labels = set(_a['mid_labels']) - {'work'}
    _bld_labels = set(_b['mid_labels'].split(';')) - {'work'}
    _flag = '' if _poi_labels <= _bld_labels else '  <- label(s) the building does not carry: ' + ', '.join(sorted(_poi_labels - _bld_labels))
    print()
    print(f"--- {_p}  |  {_rec[_p].splitlines()[0][:90]}")
    print(f"   POI: {', '.join(_a['mid_labels']):40s} ({_a['confidence']:6s}, {_a['elapsed_s']:5.1f}s)  |  building: {_b['mid_labels']}{_flag}")
    print(f"   {_a['interpreted_type'][:110]}")
    print(f"   reason: {_a['reason'][:240]}")
_med = sample['elapsed_s'].median()
print()
print(f'  ..  at the median {_med:.1f} s per call, the {len(records):,} calls of the full run take '
      f'{len(records) * _med / 3600:,.0f} hours = {len(records) * _med / 86400:.1f} days with one request at a time')


### Launching the full run

The full run does not belong in a notebook cell: it runs for days on the Linux
server. `scripts/07_run_llm_pois.py` uses the same `run` as the cell above, on
the input file of section 2, asks every POI once, and continues where it
stopped whenever it is started again. On the server (hand-off §5): the same
`config.py` — `prompt_sha(LLM_POI_SYSTEM_PROMPT)` must print the same twelve
characters on both machines — the input parquet (checksummed), the token in
`.env`, the `capacity-final` environment; nothing else.

```bash
cd <repo>
conda activate capacity-final
python -c "from config import LLM_POI_SYSTEM_PROMPT; from lib.llm_client import prompt_sha; print(prompt_sha(LLM_POI_SYSTEM_PROMPT))"
python scripts/07_run_llm_pois.py --sample          # the sample POIs -> 07_llm_poi_sample_answers.jsonl; read them
python scripts/07_run_llm_pois.py --limit 200       # the first 200 pending calls, watch pace and failures
nohup nice -n 10 python scripts/07_run_llm_pois.py >> data/output/07_llm_poi_run.log 2>&1 &
tail -f data/output/07_llm_poi_run.log              # a status block every minute
cat  data/output/07_llm_poi_status.json             # the same block as JSON
```

Keep `--workers 1` unless the operators say otherwise. Do **not** edit
`LLM_POI_SYSTEM_PROMPT` once the full run has started: the sha changes and
every answer so far becomes invalid. The cell below reads whatever the full
run has produced so far, and says so if it has not started.


In [ ]:
if LLM_POI_ANSWERS_FILE.exists():
    _cols = answer_columns('poi_id', LLM_POI_OUTPUT_SCHEMA)
    _full = load_answers(LLM_POI_ANSWERS_FILE, _cols)
    _sha = prompt_sha(LLM_POI_SYSTEM_PROMPT)
    _ok = valid_answers(_full, _sha, 'poi_id')
    _stale = int(((_full['ok'] == True) & (_full['prompt_sha'] != _sha)).sum())
    _missing = set(records['poi_id']) - set(_ok['poi_id'])
    print(f"full run: {len(_ok):,} of {len(records):,} POIs answered under prompt {_sha}, {len(_missing):,} still to ask; "
          f"{int((_full['ok'] == False).sum()):,} failed line(s)" + (f'; {_stale:,} answers under an earlier prompt' if _stale else ''))
    if LLM_POI_STATUS_FILE.exists():
        print(LLM_POI_STATUS_FILE.read_text(encoding='utf-8')[:2000])
else:
    print('the full run has not started - launch it with scripts/07_run_llm_pois.py on the Linux server as shown above')


## 6. Assembly — after the Linux run, on Windows

Not written yet. When `07_llm_poi_answers.jsonl` is back (checksummed on both
sides), this section will:

1. read the valid answers under the pinned prompt hash `257a72e709cb`
   (`LLM_POI_RUN_PROMPT_SHA` in config) —
   `valid_answers(load_answers(LLM_POI_ANSWERS_FILE, answer_columns('poi_id',
   LLM_POI_OUTPUT_SCHEMA)), LLM_POI_RUN_PROMPT_SHA, 'poi_id')` — and assert
   they cover every `poi_id` of the input parquet — missing 0 — before
   anything is joined; failed calls are re-asked on the server with
   `--ids <poi_id,...>`;
2. put the labels on the 7,864 asked POIs: `poi_llm_labels` (the model's),
   `poi_mid_labels` (plus `work` by rule, `lib/llm_run.apply_work_rule`),
   `poi_type`, `poi_confidence`, `poi_reason`, with `building_id`, `poi_use`,
   `name` and `share_in_building` from `building_pois`. Sole occupants
   (share 1), site rows and the excluded landuse and vacant pairs are not part
   of this output: the building polygon carries them;
3. print the label clash table — per `poi_use`, the POI categories its
   building does not carry. Which label counts is decided later, so both are
   kept;
4. write `07_building_pois_classified.gpkg`: layer `building_pois` with the
   7,864 asked POIs and the columns above, layer `buildings` copied unchanged
   from step 06.

This is the only place the output file is created; it never happens on the
server. `docs/08-assembly-and-redistribution-handoff.md` has the details and
what the redistribution can use.

## Where this leaves us

Sections 1 to 5 are written. Every own pair has a record, the prompt sits in
config next to the schema it is checked against and is asserted to share its
definitions with the building prompt, the call list is the input file, and
the sample cell asks the 26 sample POIs once the prompt is confirmed. The full
run is `scripts/07_run_llm_pois.py` on the Linux server.

### Next

The prompt review gate, then the sample, then the run; then **07.6** on
Windows and **step 08**, the redistribution with `share × label`.
